In [ ]:
#9.4 Koneksi ke MongoDB dari Python

In [1]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["praktikum_bigdata"]

# Memilih (atau otomatis membuat) collection
koleksi_produk = db["produk"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'local', 'praktikum_bigdata']


In [ ]:
# 9.5 Operasi CRUD dengan PyMongo
# 9.5.1 Create — Menyisipkan Dokumen

In [2]:
# Menghapus data lama jika ada (agar percobaan kita mulai dari kondisi bersih)
koleksi_produk.delete_many({})

# insert_one() — menyisipkan satu dokumen
koleksi_produk.insert_one({
    "nama_produk": "Kopi Susu",
    "kategori": "Minuman",
    "harga": 18000,
    "stok": 120,
    "tags": ["best-seller", "dingin"]
})

# insert_many() — menyisipkan banyak dokumen sekaligus
koleksi_produk.insert_many([
    {"nama_produk": "Roti Bakar", "kategori": "Makanan", "harga": 15000, "stok": 45, "tags": ["hangat"]},
    {"nama_produk": "Es Teh", "kategori": "Minuman", "harga": 8000, "stok": 200, "tags": ["dingin", "murah"]},
    {"nama_produk": "Kentang Goreng", "kategori": "Makanan", "harga": 20000, "stok": 60, "tags": ["gurih"]},
    {"nama_produk": "Cappuccino", "kategori": "Minuman", "harga": 25000, "stok": 80, "tags": ["hangat", "best-seller"]},
])

print("Jumlah dokumen di collection 'produk':", koleksi_produk.count_documents({}))

Jumlah dokumen di collection 'produk': 5


In [ ]:
# 9.5.2 Read — Membaca Dokumen

In [3]:
# find_one() — mengambil satu dokumen pertama yang cocok
satu_produk = koleksi_produk.find_one({"kategori": "Minuman"})
print("Satu produk kategori Minuman:", satu_produk)

print()

# find() — mengambil SEMUA dokumen yang cocok (mengembalikan cursor, di-looping atau di-list())
print("Seluruh produk kategori Minuman:")
for p in koleksi_produk.find({"kategori": "Minuman"}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

print()

# Filter dengan operator perbandingan: harga lebih dari 15000
print("Produk dengan harga > 15000:")
for p in koleksi_produk.find({"harga": {"$gt": 15000}}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

Satu produk kategori Minuman: {'_id': ObjectId('6ae2f0409b2e476231531bf3'), 'nama_produk': 'Kopi Susu', 'kategori': 'Minuman', 'harga': 18000, 'stok': 120, 'tags': ['best-seller', 'dingin']}

Seluruh produk kategori Minuman:
  - Kopi Susu (Rp18,000)
  - Es Teh (Rp8,000)
  - Cappuccino (Rp25,000)

Produk dengan harga > 15000:
  - Kopi Susu (Rp18,000)
  - Kentang Goreng (Rp20,000)
  - Cappuccino (Rp25,000)


In [ ]:
# 9.5.3 Update — Memperbarui Dokumen

In [4]:
# update_one() — memperbarui SATU dokumen pertama yang cocok
koleksi_produk.update_one(
    {"nama_produk": "Es Teh"},
    {"$set": {"harga": 9000}}   # menaikkan harga Es Teh
)

# update_many() — memperbarui SEMUA dokumen yang cocok
# Contoh: mengurangi stok seluruh produk kategori Minuman sebanyak 5 (simulasi penjualan)
koleksi_produk.update_many(
    {"kategori": "Minuman"},
    {"$inc": {"stok": -5}}   # $inc mengubah nilai relatif (increment/decrement)
)

print("Setelah update:")
for p in koleksi_produk.find({}, {"_id": 0, "nama_produk": 1, "harga": 1, "stok": 1}):
    print(f"  - {p['nama_produk']}: harga=Rp{p['harga']:,}, stok={p['stok']}")

Setelah update:
  - Kopi Susu: harga=Rp18,000, stok=115
  - Roti Bakar: harga=Rp15,000, stok=45
  - Es Teh: harga=Rp9,000, stok=195
  - Kentang Goreng: harga=Rp20,000, stok=60
  - Cappuccino: harga=Rp25,000, stok=75


In [ ]:
# 9.5.4 Delete — Menghapus Dokumen

In [5]:
# delete_one() — menghapus SATU dokumen pertama yang cocok
koleksi_produk.delete_one({"nama_produk": "Kentang Goreng"})

print("Jumlah dokumen tersisa:", koleksi_produk.count_documents({}))
print("Daftar produk tersisa:", [p["nama_produk"] for p in koleksi_produk.find({}, {"nama_produk": 1, "_id": 0})])

Jumlah dokumen tersisa: 4
Daftar produk tersisa: ['Kopi Susu', 'Roti Bakar', 'Es Teh', 'Cappuccino']


In [ ]:
# 9.6 Mengintegrasikan Data MongoDB ke Spark DataFrame

In [6]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_produk.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: Tipe data khusus MongoDB (_id, dan kolom berisi list seperti "tags")
# perlu disesuaikan dulu agar Spark dapat membacanya dengan benar
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas["tags"] = df_pandas["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Langkah 4: Konversi ke Spark DataFrame
df_spark_produk = spark.createDataFrame(df_pandas)

df_spark_produk.printSchema()
df_spark_produk.show()

26/10/29 14:08:01 WARN Utils: Your hostname, yan-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/29 14:08:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 14:08:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- _id: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- tags: string (nullable = true)



+--------------------+-----------+--------+-----+----+-------------------+
|                 _id|nama_produk|kategori|harga|stok|               tags|
+--------------------+-----------+--------+-----+----+-------------------+
|6ae2f0409b2e47623...|  Kopi Susu| Minuman|18000| 115|best-seller, dingin|
|6ae2f0409b2e47623...| Roti Bakar| Makanan|15000|  45|             hangat|
|6ae2f0409b2e47623...|     Es Teh| Minuman| 9000| 195|      dingin, murah|
|6ae2f0409b2e47623...| Cappuccino| Minuman|25000|  75|hangat, best-seller|
+--------------------+-----------+--------+-----+----+-------------------+



In [7]:
from pyspark.sql.functions import avg, count

# Contoh analisis sederhana: rata-rata harga & jumlah produk per kategori
df_spark_produk.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_produk").alias("jumlah_produk")
).show()

spark.stop()
print("SparkSession ditutup.")

+--------+------------------+-------------+
|kategori|   rata_rata_harga|jumlah_produk|
+--------+------------------+-------------+
| Minuman|17333.333333333332|            3|
| Makanan|           15000.0|            1|
+--------+------------------+-------------+

SparkSession ditutup.
